# CS5494 Assignment

Runs the pipeline with `sys.executable`, so the notebook works with the local venv and on Colab.
All artefacts go to `results/`; the report source is `report/main.tex`.

In [ ]:
import os, sys, subprocess
from pathlib import Path
if 'google.colab' in sys.modules:
    project = Path('/content/CS5494AS1')
    if not (project / '.git').exists():
        subprocess.run(['git', 'clone', 'https://github.com/Minnesinger02/CS5494AS1.git', str(project)], check=True)
    os.chdir(project)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
print('Project directory:', Path.cwd())


In [ ]:
import sys, subprocess
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / 'cs5494' / 'train.py').exists(), f'run this notebook from the project folder, currently {ROOT}'
PY = sys.executable
RESULTS = ROOT / 'results'
print('interpreter:', PY)

def run(*args):
    cmd = [PY, *map(str, args)]
    print('\n$', ' '.join(cmd), flush=True)
    r = subprocess.run(cmd, cwd=ROOT, text=True, encoding='utf-8', errors='replace')
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')

try:
    import torch
    print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), 'GiB')
except ImportError:
    print('torch missing: pip install -r requirements.txt first')

### 1. Checks and autoregressive models

FVSBN and one MADE member (seed 42). The other two seeds feed the mixture.

In [ ]:
run('-m','cs5494.checks','units')
run('-m','cs5494.train', '--models', 'fvsbn', 'made', '--samples', 60000, '--test-samples', 10000,
    '--side', 28, '--hidden', 1536, '--layers', 1, '--epochs', 400, '--batch', 1024,
    '--lr', 0.001, '--scheduler', 'cosine', '--device', 'auto', '--workers', 4,
    '--seed', 42, '--output', RESULTS / 'autoregressive')
for seed in (43, 44):
    run('-m','cs5494.train', '--models', 'made', '--samples', 60000, '--test-samples', 10000,
        '--side', 28, '--hidden', 1536, '--layers', 1, '--epochs', 400, '--batch', 1024,
        '--lr', 0.001, '--scheduler', 'cosine', '--device', 'auto', '--workers', 4,
        '--seed', seed, '--output', RESULTS / f'made_seed{seed}')
run('-m','cs5494.evaluate','mixture', '--runs', RESULTS / 'autoregressive', RESULTS / 'made_seed43',
    RESULTS / 'made_seed44', '--out', RESULTS / 'ensemble_metrics.json')

### 2. GAN with two-timescale updates and weight averaging

In [ ]:
run('-m','cs5494.train', '--models', 'gan', '--arch', 'dcgan', '--samples', 60000, '--test-samples', 10000,
    '--epochs', 400, '--batch', 512, '--lr', 0.0002, '--lr-d', 0.00002, '--scheduler', 'cosine',
    '--ema', 0.999, '--device', 'auto', '--workers', 4, '--output', RESULTS / 'gan_ema')
run('-m','cs5494.evaluate','gan', '--runs', RESULTS / 'gan_ema', '--weights', 'raw',
    '--out', 'comparisons/gan_eval_ema.json')
run('-m','cs5494.evaluate','gan', '--runs', RESULTS / 'gan_ema', '--weights', 'ema',
    '--out', 'comparisons/gan_eval_ema_tmp.json')

### 3. Figures and the LaTeX report

Building the PDF needs XeLaTeX and Biber; skip the last two lines if they are not installed.

In [ ]:
run('-m','cs5494.figures','digits')
run('-m','cs5494.figures','variants')
print('report source:', ROOT / 'report' / 'main.tex')
# subprocess.run(['powershell','-File',str(ROOT/'report'/'build.ps1')], cwd=ROOT/'report')
# run('-m','cs5494.checks','report')